# SC H.3492 Partially Refundable EITC - Microsimulation Analysis

This notebook calculates the aggregate impacts of South Carolina H.3492 using PolicyEngine microsimulation.

## Reform Summary
- **Baseline**: SC EITC = 125% of federal EITC (non-refundable, limited by tax liability)
- **Reform**: Makes 25% of the excess SC EITC (above tax liability) refundable

## Key Formula
- If SC EITC > tax liability:
  - Non-refundable portion = tax liability
  - Refundable portion = 25% × (SC EITC - tax liability)

## Bill Reference
[SC H.3492 (126th Session, 2025-2026)](https://www.scstatehouse.gov/sess126_2025-2026/bills/3492.htm)

In [1]:
from policyengine_us import Microsimulation
from policyengine_us.reforms.states.sc.h3492.sc_h3492_eitc_refundable import sc_h3492_eitc_refundable
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 20)

# Dataset for analysis
ENHANCED_CPS = "hf://policyengine/policyengine-us-data/enhanced_cps_2024.h5"

print("PolicyEngine SC H.3492 Analysis")
print("="*50)

PolicyEngine SC H.3492 Analysis


## 1. Setup Microsimulations

In [2]:
print("Creating baseline microsimulation...")
baseline = Microsimulation(dataset=ENHANCED_CPS)

print("Creating reformed microsimulation with SC H.3492...")
reformed = Microsimulation(reform=sc_h3492_eitc_refundable, dataset=ENHANCED_CPS)

YEAR = 2025
print(f"\nAnalysis year: {YEAR}")
print(f"Dataset: Enhanced CPS 2024")

Creating baseline microsimulation...
Creating reformed microsimulation with SC H.3492...

Analysis year: 2025
Dataset: Enhanced CPS 2024


## 2. Identify South Carolina Households

Note: `.calculate()` returns MicroSeries which auto-weight on `.sum()`. We use `.values` to get raw numpy arrays when we need to do manual filtering/calculations.

In [3]:
# Get state codes and weights as raw numpy arrays
state_code = baseline.calculate("state_code", period=YEAR, map_to="household").values
household_weight = baseline.calculate("household_weight", period=YEAR).values

# SC state code filter
sc_filter = state_code == "SC"

# Count SC households (manual weighting since we're using raw arrays)
sc_households_weighted = household_weight[sc_filter].sum()
sc_households_records = sc_filter.sum()

print(f"=== South Carolina Households ===")
print(f"Records in dataset: {sc_households_records:,}")
print(f"Weighted household count: {sc_households_weighted:,.0f}")

=== South Carolina Households ===
Records in dataset: 451
Weighted household count: 2,385,658


## 3. Total Cost of the Reform

In [4]:
# Calculate household net income under both scenarios (as raw arrays)
baseline_net_income = baseline.calculate("household_net_income", period=YEAR).values
reformed_net_income = reformed.calculate("household_net_income", period=YEAR).values

# Calculate change in net income (benefit to households)
income_change = reformed_net_income - baseline_net_income

# Filter to SC households and calculate total cost (weighted sum)
sc_income_change = income_change[sc_filter]
sc_weights = household_weight[sc_filter]

# Total cost = weighted sum of increased benefits to SC households
total_cost = (sc_income_change * sc_weights).sum()

print(f"=== Total Cost of SC H.3492 ===")
print(f"Annual cost to South Carolina: ${total_cost/1e6:,.2f} million")

=== Total Cost of SC H.3492 ===
Annual cost to South Carolina: $265.94 million


## 4. Number of Beneficiary Households

In [5]:
# Households that benefit are those with positive income change
beneficiaries_mask = (income_change > 0.5) & sc_filter  # $0.50 threshold to avoid floating point issues

# Count beneficiary households (weighted)
beneficiary_households = household_weight[beneficiaries_mask].sum()
beneficiary_records = beneficiaries_mask.sum()

# Percentage of SC households that benefit
pct_benefiting = (beneficiary_households / sc_households_weighted) * 100

print(f"=== Beneficiary Households ===")
print(f"Households benefiting: {beneficiary_households:,.0f}")
print(f"Percentage of SC households: {pct_benefiting:.1f}%")
print(f"Records in dataset with benefit: {beneficiary_records:,}")

=== Beneficiary Households ===
Households benefiting: 215,580
Percentage of SC households: 9.0%
Records in dataset with benefit: 51


## 5. Average Benefit per Household

In [6]:
# Average benefit among all SC households
avg_benefit_all_sc = total_cost / sc_households_weighted

# Average benefit among beneficiaries only
beneficiary_total = (income_change[beneficiaries_mask] * household_weight[beneficiaries_mask]).sum()
avg_benefit_beneficiaries = beneficiary_total / beneficiary_households

print(f"=== Average Benefit ===")
print(f"Average benefit (all SC households): ${avg_benefit_all_sc:,.2f}")
print(f"Average benefit (beneficiaries only): ${avg_benefit_beneficiaries:,.2f}")

=== Average Benefit ===
Average benefit (all SC households): $111.47
Average benefit (beneficiaries only): $1,233.58


## 6. Winners/Losers Analysis

In [7]:
# Count winners (positive change), losers (negative change), and unchanged
winners_mask = (income_change > 0.5) & sc_filter
losers_mask = (income_change < -0.5) & sc_filter
unchanged_mask = (np.abs(income_change) <= 0.5) & sc_filter

winners_count = household_weight[winners_mask].sum()
losers_count = household_weight[losers_mask].sum()
unchanged_count = household_weight[unchanged_mask].sum()

print(f"=== Winners/Losers Analysis ===")
print(f"Winners (gain > $0.50):    {winners_count:>12,.0f} ({winners_count/sc_households_weighted*100:>5.1f}%)")
print(f"Losers (loss > $0.50):     {losers_count:>12,.0f} ({losers_count/sc_households_weighted*100:>5.1f}%)")
print(f"Unchanged:                 {unchanged_count:>12,.0f} ({unchanged_count/sc_households_weighted*100:>5.1f}%)")
print(f"Total SC households:       {sc_households_weighted:>12,.0f}")

# This reform should only have winners (or unchanged) - no losers expected
if losers_count < 100:  # Allow for small floating point issues
    print("\n*** As expected, this reform has no losers - it only adds a refundable credit ***")

=== Winners/Losers Analysis ===
Winners (gain > $0.50):         215,580 (  9.0%)
Losers (loss > $0.50):                0 (  0.0%)
Unchanged:                    2,170,078 ( 91.0%)
Total SC households:          2,385,658

*** As expected, this reform has no losers - it only adds a refundable credit ***


## 7. Impact by Income Bracket

In [8]:
# Get household income for bracketing (as raw array)
household_income = baseline.calculate("household_net_income", period=YEAR).values

# Define income brackets
brackets = [
    (0, 25_000, "Under $25k"),
    (25_000, 50_000, "$25k-$50k"),
    (50_000, 75_000, "$50k-$75k"),
    (75_000, 100_000, "$75k-$100k"),
    (100_000, float('inf'), "Over $100k")
]

bracket_data = []
for lower, upper, label in brackets:
    # Filter to SC households in this bracket
    bracket_mask = (household_income >= lower) & (household_income < upper) & sc_filter
    
    # Count households (weighted)
    hh_count = household_weight[bracket_mask].sum()
    
    # Total benefit (weighted)
    bracket_benefit = (income_change[bracket_mask] * household_weight[bracket_mask]).sum()
    
    # Beneficiaries in bracket
    beneficiary_bracket = (income_change > 0.5) & bracket_mask
    beneficiary_count = household_weight[beneficiary_bracket].sum()
    
    # Average benefit among beneficiaries
    avg_benefit = bracket_benefit / beneficiary_count if beneficiary_count > 0 else 0
    
    bracket_data.append({
        "Income Bracket": label,
        "SC Households": f"{hh_count:,.0f}",
        "Beneficiaries": f"{beneficiary_count:,.0f}",
        "% Benefiting": f"{beneficiary_count/hh_count*100:.1f}%" if hh_count > 0 else "0.0%",
        "Total Benefit": f"${bracket_benefit/1e6:,.2f}M",
        "Avg Benefit": f"${avg_benefit:,.0f}"
    })

df_brackets = pd.DataFrame(bracket_data)
print("=== Impact by Income Bracket ===")
print(df_brackets.to_string(index=False))

=== Impact by Income Bracket ===
Income Bracket SC Households Beneficiaries % Benefiting Total Benefit Avg Benefit
    Under $25k       292,647         1,307         0.4%        $0.65M        $501
     $25k-$50k       484,915       124,043        25.6%      $160.47M      $1,294
     $50k-$75k       511,027        33,115         6.5%       $18.94M        $572
    $75k-$100k       500,246        54,216        10.8%       $83.17M      $1,534
    Over $100k       596,548         2,899         0.5%        $2.70M        $931


## 8. Poverty Impact Analysis

In [9]:
# Calculate poverty status (as raw arrays)
baseline_in_poverty = baseline.calculate("in_poverty", period=YEAR, map_to="person").values.astype(float)
reformed_in_poverty = reformed.calculate("in_poverty", period=YEAR, map_to="person").values.astype(float)
person_weight = baseline.calculate("person_weight", period=YEAR).values

# Get state code at person level for filtering
person_state = baseline.calculate("state_code", period=YEAR, map_to="person").values
sc_person_filter = person_state == "SC"

# Calculate poverty rates for SC
sc_person_weights = person_weight[sc_person_filter]
sc_total_people = sc_person_weights.sum()

baseline_poverty_rate = (baseline_in_poverty[sc_person_filter] * sc_person_weights).sum() / sc_total_people
reformed_poverty_rate = (reformed_in_poverty[sc_person_filter] * sc_person_weights).sum() / sc_total_people

poverty_rate_change = reformed_poverty_rate - baseline_poverty_rate
poverty_reduction_pct = (poverty_rate_change / baseline_poverty_rate) * 100 if baseline_poverty_rate > 0 else 0

# People lifted out of poverty
people_lifted = (baseline_in_poverty[sc_person_filter] - reformed_in_poverty[sc_person_filter]) * sc_person_weights
people_lifted_count = people_lifted[people_lifted > 0].sum()

print(f"=== Poverty Impact (Overall) ===")
print(f"SC Population: {sc_total_people:,.0f}")
print(f"Baseline poverty rate: {baseline_poverty_rate*100:.2f}%")
print(f"Reformed poverty rate: {reformed_poverty_rate*100:.2f}%")
print(f"Change in poverty rate: {poverty_rate_change*100:+.3f} pp")
print(f"Relative reduction: {poverty_reduction_pct:+.2f}%")
print(f"People lifted out of poverty: {people_lifted_count:,.0f}")

=== Poverty Impact (Overall) ===
SC Population: 5,323,807
Baseline poverty rate: 15.53%
Reformed poverty rate: 15.07%
Change in poverty rate: -0.463 pp
Relative reduction: -2.98%
People lifted out of poverty: 24,661


In [10]:
# Child poverty analysis
age = baseline.calculate("age", period=YEAR).values
is_child = age < 18

sc_child_filter = sc_person_filter & is_child
sc_child_weights = person_weight[sc_child_filter]
sc_total_children = sc_child_weights.sum()

baseline_child_poverty_rate = (baseline_in_poverty[sc_child_filter] * sc_child_weights).sum() / sc_total_children
reformed_child_poverty_rate = (reformed_in_poverty[sc_child_filter] * sc_child_weights).sum() / sc_total_children

child_poverty_change = reformed_child_poverty_rate - baseline_child_poverty_rate
child_poverty_reduction_pct = (child_poverty_change / baseline_child_poverty_rate) * 100 if baseline_child_poverty_rate > 0 else 0

# Children lifted out of poverty
children_lifted = (baseline_in_poverty[sc_child_filter] - reformed_in_poverty[sc_child_filter]) * sc_child_weights
children_lifted_count = children_lifted[children_lifted > 0].sum()

print(f"\n=== Child Poverty Impact ===")
print(f"SC Children (under 18): {sc_total_children:,.0f}")
print(f"Baseline child poverty rate: {baseline_child_poverty_rate*100:.2f}%")
print(f"Reformed child poverty rate: {reformed_child_poverty_rate*100:.2f}%")
print(f"Change in child poverty rate: {child_poverty_change*100:+.3f} pp")
print(f"Relative reduction: {child_poverty_reduction_pct:+.2f}%")
print(f"Children lifted out of poverty: {children_lifted_count:,.0f}")


=== Child Poverty Impact ===
SC Children (under 18): 1,133,734
Baseline child poverty rate: 12.12%
Reformed child poverty rate: 11.03%
Change in child poverty rate: -1.088 pp
Relative reduction: -8.97%
Children lifted out of poverty: 12,331


## 9. Visualization: Impact by Income Bracket

In [11]:
# Prepare data for visualization
viz_data = []
for lower, upper, label in brackets:
    bracket_mask = (household_income >= lower) & (household_income < upper) & sc_filter
    hh_count = household_weight[bracket_mask].sum()
    bracket_benefit = (income_change[bracket_mask] * household_weight[bracket_mask]).sum()
    beneficiary_bracket = (income_change > 0.5) & bracket_mask
    beneficiary_count = household_weight[beneficiary_bracket].sum()
    avg_benefit = bracket_benefit / beneficiary_count if beneficiary_count > 0 else 0
    
    viz_data.append({
        "bracket": label,
        "total_benefit_millions": bracket_benefit / 1e6,
        "avg_benefit": avg_benefit,
        "beneficiaries": beneficiary_count,
        "pct_benefiting": beneficiary_count / hh_count * 100 if hh_count > 0 else 0
    })

viz_df = pd.DataFrame(viz_data)

# Create subplot figure
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Benefit by Income Bracket',
        'Average Benefit per Beneficiary',
        'Number of Beneficiary Households',
        'Percentage of Households Benefiting'
    )
)

# Total benefit
fig.add_trace(
    go.Bar(x=viz_df['bracket'], y=viz_df['total_benefit_millions'],
           marker_color='#2C6496', name='Total Benefit'),
    row=1, col=1
)

# Average benefit
fig.add_trace(
    go.Bar(x=viz_df['bracket'], y=viz_df['avg_benefit'],
           marker_color='#4CAF50', name='Avg Benefit'),
    row=1, col=2
)

# Beneficiaries
fig.add_trace(
    go.Bar(x=viz_df['bracket'], y=viz_df['beneficiaries'],
           marker_color='#FF9800', name='Beneficiaries'),
    row=2, col=1
)

# Percent benefiting
fig.add_trace(
    go.Bar(x=viz_df['bracket'], y=viz_df['pct_benefiting'],
           marker_color='#9C27B0', name='% Benefiting'),
    row=2, col=2
)

# Update layout
fig.update_layout(
    height=700,
    showlegend=False,
    title_text="SC H.3492 Impact by Income Bracket"
)

fig.update_yaxes(title_text="$ Millions", row=1, col=1)
fig.update_yaxes(title_text="$ per Household", row=1, col=2)
fig.update_yaxes(title_text="Households", row=2, col=1)
fig.update_yaxes(title_text="Percent", row=2, col=2)

fig.show()

## 10. Summary Statistics

In [12]:
print("="*70)
print("SC H.3492 PARTIALLY REFUNDABLE EITC - SUMMARY")
print("="*70)
print(f"\nFISCAL IMPACT:")
print(f"  Annual cost: ${total_cost/1e6:,.2f} million")
print(f"\nBENEFICIARY ANALYSIS:")
print(f"  SC households benefiting: {beneficiary_households:,.0f}")
print(f"  Percent of SC households: {pct_benefiting:.1f}%")
print(f"  Average benefit (beneficiaries): ${avg_benefit_beneficiaries:,.2f}")
print(f"\nPOVERTY IMPACT:")
print(f"  Overall poverty rate change: {poverty_rate_change*100:+.3f} pp")
print(f"  Child poverty rate change: {child_poverty_change*100:+.3f} pp")
print(f"  People lifted out of poverty: {people_lifted_count:,.0f}")
print(f"  Children lifted out of poverty: {children_lifted_count:,.0f}")
print(f"\nKEY INSIGHT:")
print(f"  This reform primarily benefits low-income working families")
print(f"  whose SC EITC exceeds their state tax liability.")
print("="*70)

SC H.3492 PARTIALLY REFUNDABLE EITC - SUMMARY

FISCAL IMPACT:
  Annual cost: $265.94 million

BENEFICIARY ANALYSIS:
  SC households benefiting: 215,580
  Percent of SC households: 9.0%
  Average benefit (beneficiaries): $1,233.58

POVERTY IMPACT:
  Overall poverty rate change: -0.463 pp
  Child poverty rate change: -1.088 pp
  People lifted out of poverty: 24,661
  Children lifted out of poverty: 12,331

KEY INSIGHT:
  This reform primarily benefits low-income working families
  whose SC EITC exceeds their state tax liability.


## 11. Export Results

In [13]:
# Create summary DataFrame
summary_stats = {
    "Metric": [
        "Annual Cost ($ millions)",
        "Beneficiary Households",
        "Percent of SC Households Benefiting",
        "Average Benefit (all SC households)",
        "Average Benefit (beneficiaries only)",
        "Overall Poverty Rate Change (pp)",
        "Child Poverty Rate Change (pp)",
        "People Lifted Out of Poverty",
        "Children Lifted Out of Poverty"
    ],
    "Value": [
        f"{total_cost/1e6:.2f}",
        f"{beneficiary_households:,.0f}",
        f"{pct_benefiting:.1f}%",
        f"${avg_benefit_all_sc:.2f}",
        f"${avg_benefit_beneficiaries:.2f}",
        f"{poverty_rate_change*100:+.3f}",
        f"{child_poverty_change*100:+.3f}",
        f"{people_lifted_count:,.0f}",
        f"{children_lifted_count:,.0f}"
    ]
}

df_summary = pd.DataFrame(summary_stats)
print("Summary Statistics:")
print(df_summary.to_string(index=False))

# Export to CSV
df_summary.to_csv("sc_h3492_summary_stats.csv", index=False)
df_brackets.to_csv("sc_h3492_income_brackets.csv", index=False)

print("\nExported files:")
print("  - sc_h3492_summary_stats.csv")
print("  - sc_h3492_income_brackets.csv")

Summary Statistics:
                              Metric    Value
            Annual Cost ($ millions)   265.94
              Beneficiary Households  215,580
 Percent of SC Households Benefiting     9.0%
 Average Benefit (all SC households)  $111.47
Average Benefit (beneficiaries only) $1233.58
    Overall Poverty Rate Change (pp)   -0.463
      Child Poverty Rate Change (pp)   -1.088
        People Lifted Out of Poverty   24,661
      Children Lifted Out of Poverty   12,331

Exported files:
  - sc_h3492_summary_stats.csv
  - sc_h3492_income_brackets.csv
